# Decision Tree Classifier with Employee Attrition Dataset

In this notebook, we will build a decision tree classifier using the scikit-learn library. We will use a hypothetical employee attrition dataset for this example.

## Import Libraries
First, let's import the necessary libraries.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

## Load and Explore the Dataset
Next, we will load the employee attrition dataset and explore its contents.

In [3]:
# Load the dataset
df = pd.read_csv('employee_attrition.csv')

# Display the first few rows of the DataFrame
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,EducationField,Gender,HourlyRate,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,TotalWorkingYears
0,41,Yes,Travel_Rarely,1102,Sales,Life Sciences,Female,94,Sales Executive,4,Single,5993,19479,8,Yes,8
1,49,No,Travel_Frequently,279,Research & Development,Life Sciences,Male,61,Research Scientist,2,Married,5130,24907,1,No,10
2,37,Yes,Travel_Rarely,1373,Research & Development,Other,Male,92,Laboratory Technician,3,Single,2090,2396,6,Yes,7
3,33,No,Travel_Frequently,1392,Research & Development,Life Sciences,Female,56,Research Scientist,3,Married,2909,23159,1,Yes,8
4,27,No,Travel_Rarely,591,Research & Development,Medical,Male,40,Laboratory Technician,2,Married,3468,16632,9,No,6


## Preprocess the Data
We need to preprocess the data, including handling categorical variables and missing values.

In [4]:
# For simplicity, let's assume there are no missing values in this example.
# If there are, you can handle them with techniques like imputation or dropping rows/columns.

from sklearn.preprocessing import LabelEncoder

#Perform conversion of categorical variables
# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the columns that does not have values
columns_to_convert = ['BusinessTravel','Department','EducationField','Gender','JobRole','JobSatisfaction','MaritalStatus','OverTime']
for column in columns_to_convert:
    df[column] = label_encoder.fit_transform(df[column])
df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,EducationField,Gender,HourlyRate,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,TotalWorkingYears
0,41,Yes,2,1102,2,1,0,94,7,3,2,5993,19479,8,1,8
1,49,No,1,279,1,1,1,61,6,1,1,5130,24907,1,0,10
2,37,Yes,2,1373,1,4,1,92,2,2,2,2090,2396,6,1,7
3,33,No,1,1392,1,1,0,56,6,2,1,2909,23159,1,1,8
4,27,No,2,591,1,3,1,40,2,1,1,3468,16632,9,0,6


## Split the Dataset
We will split the dataset into training and testing sets.

In [5]:
# Split the dataset into training and testing sets
y = df['Attrition']
# drop the outcome column
x = df.drop('Attrition', axis=1)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


##  Modify the code below to train the Decision Tree Classifier and leverage MLFlow to track 
### &emsp; 1. the experiment run 
### &emsp; 2. by logging hyperparameters, metrics.
### &emsp; 3. Register the model

In [8]:
# Import all the required mlflow related libraries
import os
import mlflow
import mlflow.sklearn

# 重要：把 run 写到「本 notebook 所在目录」的 mlruns，这样和你在 workshop 目录启动的 mlflow ui 一致
WORKSHOP_DIR = r"d:\学工\5008\Integrating and Deploying AI Solutions\Courseware\workshop"
mlruns_path = os.path.join(WORKSHOP_DIR, "mlruns")
os.makedirs(mlruns_path, exist_ok=True)
mlflow.set_tracking_uri("file:///" + mlruns_path.replace("\\", "/"))

# Set experiment name (creates or uses existing)
mlflow.set_experiment("Employee_Attrition_DCT")

# Define at least 2 different hyperparameter sets for 2 experiments
param_sets = [
    {"max_depth": 4, "min_samples_split": 2, "random_state": 42},
    {"max_depth": 8, "min_samples_split": 5, "random_state": 42},
]

for params in param_sets:
    with mlflow.start_run():
        # Log hyperparameters
        mlflow.log_params(params)

        # Create and train the decision tree classifier
        dt_classifier = DecisionTreeClassifier(**params)
        dt_classifier.fit(X_train, y_train)

        # Make predictions on the test set
        y_pred_dt = dt_classifier.predict(X_test)

        # Calculate the accuracy of the decision tree model
        accuracy_dt = accuracy_score(y_test, y_pred_dt)

        # Log the accuracy metric
        mlflow.log_metric("accuracy", accuracy_dt)

        # Log the model and register it (same registered model name → version 1, 2, ...)
        mlflow.sklearn.log_model(
            dt_classifier,
            artifact_path="model",
            registered_model_name="EmployeeAttrition-DCT",
        )

        print(f"Run with max_depth={params['max_depth']}, min_samples_split={params['min_samples_split']} -> Accuracy: {accuracy_dt:.4f}")

c:\Users\12581\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/17 12:49:03 INFO mlflow.tracking.fluent: Experiment with name 'Employee_Attrition_DCT' does not exist. Creating a new experiment.
2026/03/17 12:49:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 12:49:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recom

Run with max_depth=4, min_samples_split=2 -> Accuracy: 0.8639
Run with max_depth=8, min_samples_split=5 -> Accuracy: 0.8333


Registered model 'EmployeeAttrition-DCT' already exists. Creating a new version of this model...
Created version '2' of model 'EmployeeAttrition-DCT'.


##  Load the Registered model and version
###  &emsp;Try to make a prediction with this loaded model

In [9]:
# Load the registered model by name and version (e.g. version 1)
# Use "models:/<model_name>/<version>" or "models:/<model_name>/latest"
loaded_model = mlflow.sklearn.load_model("models:/EmployeeAttrition-DCT/1")

# Make a prediction with the loaded model (use one row from test set)
sample = X_test.head(1)
prediction = loaded_model.predict(sample)
print("Sample features (one row):")
print(sample)
print(f"\nPrediction (Attrition): {prediction}")

Sample features (one row):
      Age  BusinessTravel  DailyRate  Department  EducationField  Gender  \
1041   28               2        866           2               3       1   

      HourlyRate  JobRole  JobSatisfaction  MaritalStatus  MonthlyIncome  \
1041          84        7                0              2           8463   

      MonthlyRate  NumCompaniesWorked  OverTime  TotalWorkingYears  
1041        23490                   0         0                  6  

Prediction (Attrition): ['No']
